In [1]:
import openai
import instructor
import pandas as pd
import cohere as co

from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Filter,
    FieldCondition,
    MatchValue,
    VectorParams,
    Distance,
    SparseVectorParams,
    Modifier,
    PayloadSchemaType,
    Document,
    PointStruct,
    Prefetch,
    FusionQuery,
    RrfQuery,
    Rrf
)

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

### Retrieval

In [4]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [5]:
def retrieve_data(query, qdrant_client, k=5):
   
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25",
                ),
                using="bm25",
                limit=20
            )
        ],
        query=RrfQuery(rrf=Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [11]:
QUERY = "Are there books for self improvement?"

In [ ]:
result = retrieve_data(QUERY, qdrant_client, k=20)

In [7]:
result

{'retrieved_context_ids': ['B0B2TW64X7',
  'B0BXN7F5Z6',
  '1953156614',
  'B0BM3VQD48',
  '1401971903',
  '8409504545',
  '1942121628',
  '0578376490',
  '057831679X',
  '1990566359',
  '195780761X',
  '1803614625',
  '1958714712',
  '1789047811',
  '3982279941',
  '1665304367',
  '1961970023',
  '1684261627',
  '1739276302',
  '1915036747'],
 'retrieved_context': ['Becoming the Change: Five Essential Elements to Being Your Best Self Becoming The Change provides the reader with the tools and resources necessary to manage the one thing we can truly control—ourselves. Wolfe discusses change much like the metamorphosis of a butterﬂy, whereas we experience our own type of metamorphosis as it relates to our own social and emotional well-being. Five innate qualities are instilled deep within our core, our moral compass, known as Social Emotional Learning (SEL). At its center is Self-Awareness which directly impacts the four directional pathways of Self-Management, Social Awareness, Relation

### Reranking

In [8]:
cohere_client = co.ClientV2()

In [10]:
to_rerank = result["retrieved_context"]

In [12]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=QUERY,
    documents=to_rerank,
    top_n=20
)

In [13]:
response

V2RerankResponse(id='cb377ca3-0e5f-413d-bf3c-ea40998025b9', results=[V2RerankResponseResultsItem(index=4, relevance_score=0.85805386), V2RerankResponseResultsItem(index=0, relevance_score=0.85709965), V2RerankResponseResultsItem(index=2, relevance_score=0.80843043), V2RerankResponseResultsItem(index=5, relevance_score=0.8072176), V2RerankResponseResultsItem(index=8, relevance_score=0.80354404), V2RerankResponseResultsItem(index=19, relevance_score=0.80168754), V2RerankResponseResultsItem(index=17, relevance_score=0.7954039), V2RerankResponseResultsItem(index=12, relevance_score=0.7928494), V2RerankResponseResultsItem(index=6, relevance_score=0.7889737), V2RerankResponseResultsItem(index=11, relevance_score=0.7889737), V2RerankResponseResultsItem(index=1, relevance_score=0.7770292), V2RerankResponseResultsItem(index=18, relevance_score=0.747321), V2RerankResponseResultsItem(index=10, relevance_score=0.7428696), V2RerankResponseResultsItem(index=7, relevance_score=0.73685527), V2RerankRe

In [14]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [15]:
reranked_results

['The Greatness Mindset: Unlock the Power of Your Mind and Live Your Best Life Today Greatness is inside you.Now is the time to wake it up. Are you living your most authentic life? Are you leaning into your purpose or running away from it? Is this the story you want your future self to tell or do you ache for something more?Through his breakthrough discoveries, New York Times best-selling author Lewis Howes reveals how you can rewrite your past to propel yourself into a powerful and abundant future.With these raw and revealing personal stories, science-backed strategies from industry-leading experts, and step-by-step guidance, you will learn how to: Clearly define a Meaningful Mission to enhance your purpose for this season of life Clearly define a Meaningful Mission to enhance your purpose for this season of life Identify the root causes of self-doubt and conquer the fears that hold you back Identify the root causes of self-doubt and conquer the fears that hold you back Transform your